# 16.6 - LLMOps

Status: VERIFIED

## What Are We Solving?

LLMs are expensive, non-deterministic, and hard to debug. LLMOps brings engineering discipline to LLM applications: versioning prompts, tracking costs, caching responses, and evaluating quality systematically.

## Mental Model

LLMOps is like running a restaurant with a celebrity chef: the chef (LLM) is brilliant but unpredictable and expensive. You need a reservation system (caching), a menu (prompt templates), and a bill tracker (cost monitoring).

## Prompt Versioning Pattern

```python
PROMPTS = {
    'v1': {
        'system': 'You are a helpful assistant.',
        'template': 'Answer this question: {question}',
    },
    'v2': {
        'system': 'You are a concise expert assistant.',
        'template': 'Answer in 3 sentences: {question}',
    },
}
```

In [1]:
import matplotlib
matplotlib.use('Agg')

# Token cost calculation
pricing = {
    'gpt-4o': {'input_per_1k': 0.0025, 'output_per_1k': 0.01},
    'gpt-4o-mini': {'input_per_1k': 0.00015, 'output_per_1k': 0.0006},
    'claude-3-5-sonnet': {'input_per_1k': 0.003, 'output_per_1k': 0.015},
}

def estimate_cost(model, input_tokens, output_tokens):
    rates = pricing[model]
    input_cost = (input_tokens / 1000) * rates['input_per_1k']
    output_cost = (output_tokens / 1000) * rates['output_per_1k']
    return input_cost + output_cost

# Simulate a day of LLM usage
import random
random.seed(42)
daily_cost = 0
daily_requests = 0
for _ in range(100):
    model = random.choice(list(pricing.keys()))
    in_tok = random.randint(200, 2000)
    out_tok = random.randint(50, 800)
    daily_cost += estimate_cost(model, in_tok, out_tok)
    daily_requests += 1

print("Daily LLM Cost Report:")
print(f"  Total requests: {daily_requests}")
print(f"  Total cost:     ${daily_cost:.4f}")
print(f"  Avg per request: ${daily_cost/daily_requests:.6f}")


Daily LLM Cost Report:
  Total requests: 100
  Total cost:     $0.6358
  Avg per request: $0.006358


## Response Caching Pattern

In [2]:
import matplotlib
matplotlib.use('Agg')
import hashlib
import json
import time

# Simple in-memory LLM cache
class LLMCache:
    def __init__(self, ttl_seconds=3600):
        self._cache = {}
        self._ttl = ttl_seconds
        self.hits = 0
        self.misses = 0

    def _key(self, model, prompt, system_msg=''):
        raw = json.dumps({'model': model, 'prompt': prompt, 'system': system_msg})
        return hashlib.sha256(raw.encode()).hexdigest()[:16]

    def get(self, model, prompt, system_msg=''):
        key = self._key(model, prompt, system_msg)
        if key in self._cache:
            entry = self._cache[key]
            if time.time() - entry['time'] < self._ttl:
                self.hits += 1
                return entry['response']
            del self._cache[key]
        self.misses += 1
        return None

    def put(self, model, prompt, response, system_msg=''):
        key = self._key(model, prompt, system_msg)
        self._cache[key] = {'response': response, 'time': time.time()}

cache = LLMCache(ttl_seconds=600)
test_prompts = [
    ('gpt-4o', 'What is machine learning?'),
    ('gpt-4o', 'What is machine learning?'),  # duplicate
    ('gpt-4o', 'What is deep learning?'),
    ('gpt-4o', 'What is machine learning?'),  # cached
]

for model, prompt in test_prompts:
    result = cache.get(model, prompt)
    if result is None:
        fake_response = f"Response to: {prompt[:30]}..."
        cache.put(model, prompt, fake_response)
        print(f"  MISS: {prompt[:40]}")
    else:
        print(f"  HIT:  {prompt[:40]}")

print(f"\nCache stats: {cache.hits} hits, {cache.misses} misses, "
      f"hit rate: {cache.hits/(cache.hits+cache.misses):.1%}")


  MISS: What is machine learning?
  HIT:  What is machine learning?
  MISS: What is deep learning?
  HIT:  What is machine learning?

Cache stats: 2 hits, 2 misses, hit rate: 50.0%


## LLMOps Checklist

- [ ] **Prompt versioning**: every prompt has a version tag
- [ ] **Cost tracking**: log token usage per request
- [ ] **Caching**: cache deterministic/repeated queries
- [ ] **Evaluation set**: golden dataset for quality checks
- [ ] **Fallback**: handle rate limits, timeouts, refusals
- [ ] **Logging**: capture full prompt + response for debugging

In [3]:
import matplotlib
matplotlib.use('Agg')
print('VERIFICATION PASSED: Phase 16.6 complete')


VERIFICATION PASSED: Phase 16.6 complete
